# Two-head GapFinder: loss-weight ablation and detector-threshold tuning

This experiment answers two questions without changing the architecture:

1. Which combination of tail-regression weight $\alpha \in \{2,3,5\}$ and detector-loss weight $\lambda \in \{0.5,1\}$ gives the best validation trade-off?
2. Which detector threshold should replace the arbitrary default of `0.5`?

The 70/15/15 split is identical to the previous experiments. Models and thresholds are selected using **validation only**. The test set is evaluated once, after selection.

In [ ]:
from dataclasses import replace
import gc
import json
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
import torch
from transformers import set_seed

import functions
from Datasets.dataset_gap_finder import DatasetGapFinder
from Models.lora import LoRASettings
from Models.model_two_head_gap_finder import TwoHeadGapFinder
from Trainers.trainer_two_head_gap_finder import (
    TwoHeadGapFinderTrainer,
    TwoHeadGapFinderTrainingConfig,
    compute_two_head_report,
)
from functions import ConfigTrainClassifier, PolicySpec, RewardSpec

logging.getLogger().setLevel(logging.INFO)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)

## 1. Configuration and artifact discovery

`MIN_VALIDATION_RECALL=0.70` chooses the highest-F1 validation threshold that still reaches at least 70% recall. Set it to `None` to optimize validation F1 without a recall constraint. Existing completed checkpoints are reused.

In [ ]:
RANDOM_STATE = 42
MODEL_NAME = "Qwen/Qwen3-0.6B"
GRID = [(2.0, 0.5), (2.0, 1.0), (3.0, 0.5), (3.0, 1.0), (5.0, 0.5), (5.0, 1.0)]
THRESHOLDS = np.round(np.arange(0.01, 1.00, 0.01), 2)
MIN_VALIDATION_RECALL = 0.70
REUSE_EXISTING_CHECKPOINTS = True
GENERATE_DATA_IF_MISSING = True

EPOCHS = 3.0
BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 8
MAX_LENGTH = 512
LEARNING_RATE = 2e-5

search_bases = [Path.cwd(), *Path.cwd().parents]
workspace = Path("/workspace")
if workspace.is_dir():
    search_bases.extend([workspace, *[p for p in workspace.iterdir() if p.is_dir()]])
candidate_roots = []
for base in search_bases:
    candidate = base / "outputs" / "gap_finder_predictability"
    if candidate not in candidate_roots:
        candidate_roots.append(candidate)

SOURCE_ROOT = next(
    (root for root in candidate_roots
     if (root / "calibration.json").is_file()
     and (root / "all_20k.json").is_file()),
    Path.cwd() / "outputs" / "gap_finder_predictability",
)
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT = SOURCE_ROOT.parent / "two_head_gap_finder_ablation"
EXISTING_MODEL_ROOT = SOURCE_ROOT.parent / "two_head_gap_finder"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Kernel working directory: {Path.cwd()}")
print(f"Using data from: {SOURCE_ROOT.resolve()}")
print(f"Writing experiment to: {OUTPUT_ROOT.resolve()}")

In [ ]:
calibration_path = SOURCE_ROOT / "calibration.json"
dataset_path = SOURCE_ROOT / "all_20k.json"
if calibration_path.is_file() and dataset_path.is_file():
    gap_calibration = functions.GapCalibration.load(calibration_path)
    gap_dataset = DatasetGapFinder.load(dataset_path)
    print("Loaded existing calibration and 20k dataset.")
else:
    if not GENERATE_DATA_IF_MISSING:
        raise FileNotFoundError(f"Missing {calibration_path} or {dataset_path}")
    print("Artifacts are genuinely absent; generating the calibration and 20k dataset.")
    policy_spec = PolicySpec(model_name=MODEL_NAME, lora_config=None)
    proxy_spec = RewardSpec(
        class_name="RewardModel", model_name="Skywork/Skywork-Reward-V2-Qwen3-0.6B",
        mode_name="proxy",
    )
    judge_spec = RewardSpec(
        class_name="RewardModel", model_name="Skywork/Skywork-Reward-V2-Qwen3-4B",
        mode_name="judge",
    )
    calibration_config = ConfigTrainClassifier(
        dataset_name="Anthropic/hh-rlhf", policy=policy_spec, reward=proxy_spec, judge=judge_spec,
        start_dataset=0, end_dataset=5_000, generation_batch_size=64,
        reward_batch_size=128, judge_batch_size=32, score_max_length=1024,
    )
    gap_data_config = replace(
        calibration_config, start_dataset=5_000, end_dataset=25_000
    )
    if calibration_path.is_file():
        gap_calibration = functions.GapCalibration.load(calibration_path)
    else:
        gap_calibration = functions.calculate_gap_calibration(calibration_config)
        gap_calibration.save(calibration_path)
    gap_dataset = functions.collect_gap_finder_dataset(gap_data_config, gap_calibration)
    gap_dataset.save(dataset_path)
if len(gap_dataset) != 20_000:
    raise ValueError(f"Expected 20,000 samples, found {len(gap_dataset):,}")

train_dataset, validation_dataset, test_dataset = gap_dataset.split_three_way(
    train_size=0.70, validation_size=0.15, test_size=0.15, random_state=RANDOM_STATE
)
assert (len(train_dataset), len(validation_dataset), len(test_dataset)) == (14_000, 3_000, 3_000)

def tail_count(dataset):
    gaps = np.asarray([row["labels"] for row in dataset.dataset])
    return int((gaps > gap_calibration.theta).sum())

pd.DataFrame({
    "samples": [len(train_dataset), len(validation_dataset), len(test_dataset)],
    "high_gap": [tail_count(train_dataset), tail_count(validation_dataset), tail_count(test_dataset)],
}, index=["train", "validation", "test"] )

## 2. Validation-threshold helpers

PR-AUC and ROC-AUC measure ranking and do not depend on a threshold. Precision, recall, F1, and the confusion matrix are recomputed for every candidate threshold.

In [ ]:
def dataset_arrays(dataset):
    prompts = [row["prompt"] for row in dataset.dataset]
    answers = [row["answer"] for row in dataset.dataset]
    gaps = np.asarray([row["labels"] for row in dataset.dataset], dtype=np.float64)
    return prompts, answers, gaps

def threshold_curve(actual_gaps, probabilities, theta):
    actual_tail = np.asarray(actual_gaps) > theta
    rows = []
    for threshold in THRESHOLDS:
        predicted_tail = np.asarray(probabilities) >= threshold
        tn, fp, fn, tp = confusion_matrix(
            actual_tail, predicted_tail, labels=[False, True]
        ).ravel()
        rows.append({
            "threshold": float(threshold),
            "precision": float(precision_score(actual_tail, predicted_tail, zero_division=0)),
            "recall": float(recall_score(actual_tail, predicted_tail, zero_division=0)),
            "f1": float(f1_score(actual_tail, predicted_tail, zero_division=0)),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "predicted_positive": int(predicted_tail.sum()),
        })
    return pd.DataFrame(rows)

def choose_operating_point(curve, minimum_recall=MIN_VALIDATION_RECALL):
    candidates = curve
    if minimum_recall is not None:
        candidates = curve[curve["recall"] >= minimum_recall]
        if candidates.empty:
            raise ValueError(f"No threshold reaches validation recall {minimum_recall}")
    return candidates.sort_values(
        ["f1", "precision", "threshold"], ascending=[False, False, False]
    ).iloc[0]

def prefixed(report, prefix):
    return {f"{prefix}_{key}": value for key, value in report.items()}

## 3. Train the six-model grid and tune each threshold on validation

This is the expensive cell. It keeps only one model on the GPU at a time. A `final` checkpoint with valid metadata is loaded rather than retrained. Validation predictions, threshold curves, and metrics are saved under each run directory.

In [ ]:
validation_prompts, validation_answers, actual_validation_gaps = dataset_arrays(validation_dataset)
grid_rows = []

for high_gap_weight, detector_loss_weight in GRID:
    run_name = f"alpha={high_gap_weight:g}_lambda={detector_loss_weight:g}"
    run_directory = OUTPUT_ROOT / run_name
    final_checkpoint = run_directory / "final"
    metadata_path = final_checkpoint / TwoHeadGapFinder.METADATA_FILE_NAME
    existing_checkpoint = EXISTING_MODEL_ROOT / run_name / "final"
    existing_metadata = existing_checkpoint / TwoHeadGapFinder.METADATA_FILE_NAME
    print(f"\n=== {run_name} ===")

    set_seed(RANDOM_STATE)
    if REUSE_EXISTING_CHECKPOINTS and metadata_path.is_file():
        model = TwoHeadGapFinder.load(final_checkpoint)
        print(f"Loaded {final_checkpoint}")
    elif REUSE_EXISTING_CHECKPOINTS and existing_metadata.is_file():
        final_checkpoint = existing_checkpoint
        model = TwoHeadGapFinder.load(final_checkpoint)
        print(f"Reused earlier experiment checkpoint: {final_checkpoint}")
    else:
        model = TwoHeadGapFinder(
            MODEL_NAME, theta=gap_calibration.theta,
            gap_finder_id=gap_dataset.id, source_policy=MODEL_NAME,
        )
        config = TwoHeadGapFinderTrainingConfig(
            output_dir=str(run_directory), theta=gap_calibration.theta,
            epochs=EPOCHS, batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            gradient_checkpointing=True, learning_rate=LEARNING_RATE,
            max_length=MAX_LENGTH, high_gap_weight=high_gap_weight,
            detector_loss_weight=detector_loss_weight,
            lora_settings=LoRASettings(),
        )
        TwoHeadGapFinderTrainer(model, config).train(train_dataset, validation_dataset)

    predicted_gaps, probabilities = model.predict(
        validation_prompts, validation_answers,
        batch_size=BATCH_SIZE, max_length=MAX_LENGTH,
    )
    predicted_gaps = np.asarray(predicted_gaps)
    probabilities = np.asarray(probabilities)
    curve = threshold_curve(actual_validation_gaps, probabilities, gap_calibration.theta)
    operating_point = choose_operating_point(curve)
    tuned_threshold = float(operating_point["threshold"])

    fixed_report = compute_two_head_report(
        actual_validation_gaps, predicted_gaps, probabilities,
        theta=gap_calibration.theta, detector_threshold=0.5,
    )
    tuned_report = compute_two_head_report(
        actual_validation_gaps, predicted_gaps, probabilities,
        theta=gap_calibration.theta, detector_threshold=tuned_threshold,
    )
    row = {
        "run_name": run_name,
        "high_gap_weight": high_gap_weight,
        "detector_loss_weight": detector_loss_weight,
        "checkpoint_path": str(final_checkpoint.resolve()),
        **prefixed(fixed_report, "fixed_0_5"),
        **prefixed(tuned_report, "tuned"),
    }
    grid_rows.append(row)

    run_directory.mkdir(parents=True, exist_ok=True)
    curve.to_csv(run_directory / "validation_threshold_curve.csv", index=False)
    np.savez_compressed(
        run_directory / "validation_predictions.npz",
        actual_gaps=actual_validation_gaps, predicted_gaps=predicted_gaps,
        detector_probabilities=probabilities,
    )
    (run_directory / "validation_report.json").write_text(
        json.dumps(row, indent=2, sort_keys=True) + "\n"
    )
    print({
        "threshold": tuned_threshold,
        "precision": tuned_report["detector_precision"],
        "recall": tuned_report["detector_recall"],
        "f1": tuned_report["detector_f1"],
        "mse": tuned_report["mse"],
        "pr_auc": tuned_report["detector_pr_auc"],
    })

    model.offload()
    del model, predicted_gaps, probabilities
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

validation_results = pd.DataFrame(grid_rows)
validation_results.to_csv(OUTPUT_ROOT / "validation_grid_results.csv", index=False)
validation_results

## 4. Select the configuration using validation only

The rule is fixed before viewing test labels: maximize tuned validation F1 subject to the recall floor, break ties by PR-AUC, then by lower MSE.

In [ ]:
ranking = validation_results.sort_values(
    ["tuned_detector_f1", "tuned_detector_pr_auc", "tuned_mse"],
    ascending=[False, False, True],
).reset_index(drop=True)
selected = ranking.iloc[0]
display_columns = [
    "run_name", "tuned_detector_threshold", "tuned_detector_precision",
    "tuned_detector_recall", "tuned_detector_f1",
    "tuned_detector_pr_auc", "tuned_detector_roc_auc",
    "tuned_mse", "tuned_r2", "tuned_mae_d_gt_theta",
]
print(f"Selected using validation: {selected['run_name']}")
ranking[display_columns]

In [ ]:
selected_directory = OUTPUT_ROOT / selected["run_name"]
selected_curve = pd.read_csv(selected_directory / "validation_threshold_curve.csv")
selected_predictions = np.load(selected_directory / "validation_predictions.npz")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(selected_curve["threshold"], selected_curve["precision"], label="precision")
axes[0].plot(selected_curve["threshold"], selected_curve["recall"], label="recall")
axes[0].plot(selected_curve["threshold"], selected_curve["f1"], label="F1")
axes[0].axvline(selected["tuned_detector_threshold"], color="black", linestyle="--", label="selected")
axes[0].set(xlabel="Detector threshold", ylabel="Validation metric", ylim=(0, 1))
axes[0].legend()

actual_tail = selected_predictions["actual_gaps"] > gap_calibration.theta
precision, recall, _ = precision_recall_curve(
    actual_tail, selected_predictions["detector_probabilities"]
)
axes[1].plot(recall, precision)
axes[1].axhline(actual_tail.mean(), color="gray", linestyle=":", label="random baseline")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Validation precision-recall curve", xlim=(0, 1), ylim=(0, 1))
axes[1].legend()
plt.tight_layout()

## 5. Evaluate the selected model once on the untouched test set

The detector threshold below is copied from validation; it is not retuned on test. Both `0.5` and the selected operating point are reported to quantify the benefit of threshold tuning.

In [ ]:
selected_model = TwoHeadGapFinder.load(Path(selected["checkpoint_path"]))
test_prompts, test_answers, actual_test_gaps = dataset_arrays(test_dataset)
predicted_test_gaps, test_probabilities = selected_model.predict(
    test_prompts, test_answers, batch_size=BATCH_SIZE, max_length=MAX_LENGTH
)
predicted_test_gaps = np.asarray(predicted_test_gaps)
test_probabilities = np.asarray(test_probabilities)
selected_threshold = float(selected["tuned_detector_threshold"])

fixed_test_report = compute_two_head_report(
    actual_test_gaps, predicted_test_gaps, test_probabilities,
    theta=gap_calibration.theta, detector_threshold=0.5,
)
tuned_test_report = compute_two_head_report(
    actual_test_gaps, predicted_test_gaps, test_probabilities,
    theta=gap_calibration.theta, detector_threshold=selected_threshold,
)
test_report = {
    "selected_run": selected["run_name"],
    "selection_minimum_validation_recall": MIN_VALIDATION_RECALL,
    **prefixed(fixed_test_report, "fixed_0_5"),
    **prefixed(tuned_test_report, "tuned"),
}
(OUTPUT_ROOT / "selected_test_report.json").write_text(
    json.dumps(test_report, indent=2, sort_keys=True) + "\n"
)
np.savez_compressed(
    OUTPUT_ROOT / "selected_test_predictions.npz",
    actual_gaps=actual_test_gaps, predicted_gaps=predicted_test_gaps,
    detector_probabilities=test_probabilities,
)
pd.DataFrame({
    "threshold_0.5": fixed_test_report,
    "validation_tuned": tuned_test_report,
})

In [ ]:
actual_test_tail = actual_test_gaps > gap_calibration.theta
confusions = {}
for name, threshold in [("threshold_0.5", 0.5), ("validation_tuned", selected_threshold)]:
    tn, fp, fn, tp = confusion_matrix(
        actual_test_tail, test_probabilities >= threshold, labels=[False, True]
    ).ravel()
    confusions[name] = {"TN": tn, "FP": fp, "FN": fn, "TP": tp}
pd.DataFrame(confusions).T

## 6. Compare against regression-only detection

If the previous regression report exists, this table compares its $\hat d > \theta$ detector with the selected two-head model.

In [ ]:
regression_report_path = SOURCE_ROOT / f"id={gap_dataset.id}" / "test_report.json"
regression_report = (
    json.loads(regression_report_path.read_text()) if regression_report_path.is_file() else {}
)
comparison = pd.DataFrame([
    {
        "model": "regression_only",
        "mse": regression_report.get("mse"),
        "r2": regression_report.get("r2"),
        "tail_mae": regression_report.get("mae_d_gt_theta"),
        "precision": regression_report.get("precision_d_gt_theta"),
        "recall": regression_report.get("recall_d_gt_theta"),
        "f1": regression_report.get("f1_d_gt_theta"),
        "pr_auc": None, "roc_auc": None, "threshold": gap_calibration.theta,
    },
    {
        "model": "two_head_fixed_0.5",
        "mse": fixed_test_report["mse"], "r2": fixed_test_report["r2"],
        "tail_mae": fixed_test_report["mae_d_gt_theta"],
        "precision": fixed_test_report["detector_precision"],
        "recall": fixed_test_report["detector_recall"],
        "f1": fixed_test_report["detector_f1"],
        "pr_auc": fixed_test_report["detector_pr_auc"],
        "roc_auc": fixed_test_report["detector_roc_auc"], "threshold": 0.5,
    },
    {
        "model": "two_head_validation_tuned",
        "mse": tuned_test_report["mse"], "r2": tuned_test_report["r2"],
        "tail_mae": tuned_test_report["mae_d_gt_theta"],
        "precision": tuned_test_report["detector_precision"],
        "recall": tuned_test_report["detector_recall"],
        "f1": tuned_test_report["detector_f1"],
        "pr_auc": tuned_test_report["detector_pr_auc"],
        "roc_auc": tuned_test_report["detector_roc_auc"], "threshold": selected_threshold,
    },
]).set_index("model")
comparison.to_csv(OUTPUT_ROOT / "selected_test_comparison.csv")
comparison

## Reading the result

A useful outcome is not necessarily the lowest global MSE. For this project, look for a threshold-tuned detector that removes many false positives while maintaining the chosen recall floor, together with acceptable tail MAE. PR-AUC should be compared with the approximately 5% positive-class baseline. ROC-AUC measures ranking quality but can look optimistic under strong class imbalance, so report both.